In [1]:
import tkinter as tk

root = tk.Tk()
root.title("Repaso: mostrador")
root.geometry("380x160")
contador = 0

def saludar():
    global contador
    contador += 1
    etiqueta.config(text=f"Pedidos atendidos: {contador}")

etiqueta = tk.Label(root, text="Pedidos atendidos: 0")
etiqueta.pack(pady=16)
tk.Button(root, text="Atender pedido", command=saludar).pack()
root.mainloop()

In [2]:
import tkinter as tk
from tkinter import messagebox

root = tk.Tk()
root.title("Pedido: entradas")
root.geometry("460x340")
nombre_var = tk.StringVar()

tk.Label(root, text="Nombre del cliente:").pack(anchor="w", padx=12, pady=(12, 2))
tk.Entry(root, textvariable=nombre_var, width=40).pack(anchor="w", padx=12)
tk.Label(root, text="Detalle del pedido:").pack(anchor="w", padx=12, pady=(12, 2))
detalle = tk.Text(root, height=6, width=48)
detalle.pack(padx=12)

def mostrar():
    nombre = nombre_var.get().strip()
    pedido = detalle.get("1.0", "end-1c").strip()
    if not nombre or not pedido:
        messagebox.showwarning("Datos incompletos", "Escribe nombre y detalle.")
        return
    messagebox.showinfo("Pedido", f"Cliente: {nombre}\nDetalle: {pedido}")

tk.Button(root, text="Ver resumen", command=mostrar).pack(pady=12)
root.mainloop()

In [7]:
import tkinter as tk
from tkinter import ttk, messagebox

root = tk.Tk()
root.title("Opciones del pedido")
root.geometry("400x310")
turno = tk.StringVar(value="Mañana")
entrega = tk.BooleanVar(value=False)
categoria = tk.StringVar(value="Seleccionar...")

tk.Label(root, text="Turno:").pack(anchor="w", padx=12, pady=(12, 0))
for opcion in ("Mañana", "Tarde", "Noche"):
    tk.Radiobutton(root, text=opcion, variable=turno, value=opcion).pack(anchor="w", padx=12)
tk.Checkbutton(root, text="Entrega a domicilio", variable=entrega).pack(anchor="w", padx=12, pady=8)
tk.Label(root, text="Categoría:").pack(anchor="w", padx=12)
combo = ttk.Combobox(root, textvariable=categoria, values=("Higiene", "Cuidado personal", "Otros"), state="readonly")
combo.pack(anchor="w", padx=12, pady=4)

def confirmar():
    if categoria.get() == "Seleccionar...":
        messagebox.showwarning("Falta una categoría", "Selecciona una categoría.")
        return
    messagebox.showinfo("Resumen", f"Turno: {turno.get()}\nEntrega: {'Sí' if entrega.get() else 'No'}\nCategoría: {categoria.get()}")

tk.Button(root, text="Confirmar", command=confirmar).pack(pady=12)
root.mainloop()

In [9]:
import tkinter as tk
from tkinter import filedialog, messagebox

root = tk.Tk()
root.title("Editor local")
root.geometry("520x340")
texto = tk.Text(root, wrap="word")
texto.pack(fill="both", expand=True)

def abrir():
    ruta = filedialog.askopenfilename(filetypes=[("Texto", "*.txt"), ("Todos", "*.*")])
    if not ruta:
        return
    try:
        with open(ruta, "r", encoding="utf-8") as archivo:
            contenido = archivo.read()
    except (OSError, UnicodeError) as error:
        messagebox.showerror("No se pudo abrir", str(error))
        return
    texto.delete("1.0", "end")
    texto.insert("1.0", contenido)

def guardar():
    ruta = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Texto", "*.txt")])
    if not ruta:
        return
    try:
        with open(ruta, "w", encoding="utf-8") as archivo:
            archivo.write(texto.get("1.0", "end-1c"))
    except OSError as error:
        messagebox.showerror("No se pudo guardar", str(error))
        return
    messagebox.showinfo("Guardado", "Archivo guardado correctamente.")

barra = tk.Menu(root)
archivo_menu = tk.Menu(barra, tearoff=0)
archivo_menu.add_command(label="Abrir...", command=abrir)
archivo_menu.add_command(label="Guardar como...", command=guardar)
barra.add_cascade(label="Archivo", menu=archivo_menu)
root.config(menu=barra)
root.mainloop()

In [10]:

import tkinter as tk
from tkinter import filedialog, messagebox, ttk

class BlocNotas:
    def __init__(self, root):
        self.root = root
        self.root.title("Bloc de Notas - Mini Proyecto")
        self.filepath = None

        # Barra de menús
        menubar = tk.Menu(root)
        menu_archivo = tk.Menu(menubar, tearoff=0)
        menu_archivo.add_command(label="Nuevo", command=self.nuevo)
        menu_archivo.add_command(label="Abrir...", command=self.abrir)
        menu_archivo.add_command(label="Guardar", command=self.guardar)
        menu_archivo.add_command(label="Guardar como...", command=self.guardar_como)
        menu_archivo.add_separator()
        menu_archivo.add_command(label="Salir", command=root.quit)
        menubar.add_cascade(label="Archivo", menu=menu_archivo)

        root.config(menu=menubar)

        # Área de texto con scroll
        self.text = tk.Text(root, wrap="word", undo=True)
        scroll = ttk.Scrollbar(root, command=self.text.yview)
        self.text.configure(yscrollcommand=scroll.set)
        self.text.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")

        # Barra de estado
        self.status = tk.StringVar(value="Listo")
        statusbar = ttk.Label(root, textvariable=self.status, anchor="w")
        statusbar.pack(side="bottom", fill="x")

        # Eventos
        self.text.bind("<<Modified>>", self._on_modified)

    def _on_modified(self, event):
        self.status.set("Modificado...")
        self.text.edit_modified(False)

    def nuevo(self):
        self.text.delete("1.0", "end")
        self.filepath = None
        self.status.set("Nuevo documento")

    def abrir(self):
        ruta = filedialog.askopenfilename(filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
        if ruta:
            with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
                contenido = f.read()
            self.text.delete("1.0", "end")
            self.text.insert("1.0", contenido)
            self.filepath = ruta
            self.status.set(f"Abierto: {ruta}")

    def guardar(self):
        if not self.filepath:
            return self.guardar_como()
        with open(self.filepath, "w", encoding="utf-8") as f:
            f.write(self.text.get("1.0", "end-1c"))
        self.status.set(f"Guardado: {self.filepath}")

    def guardar_como(self):
        ruta = filedialog.asksaveasfilename(defaultextension=".txt", filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
        if ruta:
            self.filepath = ruta
            self.guardar()

if __name__ == "__main__":
    root = tk.Tk()
    root.geometry("600x420")
    app = BlocNotas(root)
    root.mainloop()


## 7. Ejercicios de aplicación 

Trabaja individualmente. Los ejercicios 1–4 sirven para consolidar; 5–7 amplían el proyecto. Entrega el código `.py` o el notebook con capturas o una demostración funcional.

1. **Saludo validado (5 min):** crea `Entry`, botón y `Label`. Si el nombre está vacío, muestra un aviso; si tiene texto, saluda.
2. **Turno de atención (5 min):** crea tres `Radiobutton` y muestra la elección al pulsar “Confirmar”.
3. **Ficha de pedido (8 min):** registra cliente, observaciones en `Text` y entrega a domicilio con `Checkbutton`. Presenta un resumen validado.
4. **Categoría con combo (8 min):** usa `ttk.Combobox` con al menos cuatro categorías. Bloquea el valor libre y exige elegir una opción.
5. **Buscar con Enter (8 min):** reutiliza la lista local de productos; el botón y Enter deben llamar a la misma función. Señala cuántas coincidencias aparecen.
6. **Bloc de notas mejorado (12 min):** añade “Contar palabras” al menú y muestra el número en la barra de estado. Un archivo vacío tiene cero palabras; puedes usar `len(contenido.split())`.
7. **Reto integrador (si queda tiempo o como tarea):** antes de reemplazar un documento al abrir otro o al crear uno nuevo, consulta si debe guardarse. Si cancela, conserva el texto. Considera errores al abrir/guardar y distingue entre “Guardar” y “Guardar como”.

**Criterios de logro:** la interfaz abre y responde; valida datos incompletos; presenta resultados claros; guarda y recupera UTF-8; no modifica el contenido al cancelar. Trabaja en parejas para revisar el reto 7.

In [13]:
#Saludo validado (5 min):** crea `Entry`, botón y `Label`. 
#Si el nombre está vacío, muestra un aviso; si tiene texto, saluda.**

import tkinter as tk
from tkinter import messagebox

def saludar():
    nombre=entrada_nombre.get()
    if nombre=="":
        messagebox.showerror("error","ingrese un nombre !")
    else:
        messagebox.showinfo("Nombre",f"HOLAAAA {nombre}")

ventana=tk.Tk()
ventana.geometry("460x350")
ventana.title("SALUDO CON NOMBRE")

tk.Label(ventana, text="Ingrese nombre")

entrada_nombre=tk.Entry(ventana, width=20)
entrada_nombre.grid(row=0, column=1, padx=5, pady=5)

tk.Button(ventana, text="Saludar", command=saludar, height=10, width=10).grid(row=1, column=1, padx=5, pady=5)

ventana.mainloop()

In [29]:
#2. **Turno de atención (5 min):** crea tres `Radiobutton` y muestra la elección
# al pulsar “Confirmar”.

import tkinter as tk
from tkinter import messagebox

def mostrar():
    opcion=entrada_opcion.get()
    messagebox.showinfo("eleccion",f"la eleccion es {opcion}")


ventana=tk.Tk()
ventana.geometry("460x350")
ventana.title("mostrar eleccion")

entrada_opcion=tk.StringVar(value="ninguno")

tk.Radiobutton(ventana, text="cumbia", variable=entrada_opcion, value="cumbia" ).grid(row=1, column=1, padx=5, pady=5)
tk.Radiobutton(ventana, text="reggaeton", variable=entrada_opcion, value="reggaeton" ).grid(row=2, column=1, padx=5, pady=5)
tk.Radiobutton(ventana, text="rock", variable=entrada_opcion, value="rock" ).grid(row=3, column=1, padx=5, pady=5)

tk.Button(ventana, text="Mostrar", command=mostrar, width=8, height=2).grid(row=1, column=5, padx=5, pady=5)


ventana.mainloop()

In [ ]:
#3. **Ficha de pedido (8 min):** registra cliente, observaciones en `Text` y entrega a 
#domicilio con `Checkbutton`. Presenta un resumen validado.

import tkinter as tk
from tkinter import messagebox

def mostrar():
    domicilio=entrada_domicilio.get()
    if domicilio:
        messagebox.showinfo("Resumen","Si tiene entrega a domicilio")
    else:
        messagebox.showinfo("Resumen","NOOO tiene entrega a domicilio")

ventana=tk.Tk()



